In [ ]:
# ======================================================================
# CAPÍTULO 4 - IMPORTS, ESTILO E CORES
# ======================================================================

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuração visual profissional
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['lines.markersize'] = 8

CORES = {
    'MRT': '#E74C3C',
    'T2F': '#3498DB',
    'Fase_A': '#00CED1',
    'Fase_B': '#FF6347',
    'Fase_C': '#32CD32',
    'I0': '#4169E1',
    'I1': '#228B22',
    'I2': '#8B008B'
}


In [ ]:
# ======================================================================
# PROCESSADOR DE SINAIS (lê .mat bruto e calcula RMS + seq. simétricas)
# ======================================================================

class ProcessadorSinais:
    def __init__(self, arquivo_mat, freq=60):
        self.arquivo = arquivo_mat
        self.freq = freq
        self.dados = {}
        self._carregar_dados()
        self._processar_sinais()
    
    def _carregar_dados(self):
        with h5py.File(self.arquivo, 'r') as f:
            if 't' in f.keys():
                self.t = np.array(f['t']).flatten()
            else:
                raise ValueError("Arquivo sem vetor 't'")
            self.m1 = float(f['m1'][()]) if 'm1' in f.keys() else 0.5
            for key in f.keys():
                if key.endswith('_raw'):
                    nome = key.replace('_raw', '')
                    dados_raw = np.array(f[key])
                    if dados_raw.ndim == 1:
                        dados_raw = np.column_stack([dados_raw]*3)
                    elif dados_raw.shape[0] == 3 and dados_raw.shape[1] > 3:
                        dados_raw = dados_raw.T
                    L = min(len(self.t), len(dados_raw))
                    self.t = self.t[:L]
                    dados_raw = dados_raw[:L]
                    self.dados[nome] = {'raw': dados_raw, 'rms': None, 'seq': None}
        print(f"✓ {Path(self.arquivo).stem[:90]}")
    
    def _processar_sinais(self):
        dt = self.t[1] - self.t[0]
        self.fs = 1.0/dt
        self.samples_ciclo = max(1, int(self.fs/60))
        for nome, sinal in self.dados.items():
            raw = sinal['raw']
            try:
                sinal['rms'] = np.sqrt(np.abs(
                    scipy.ndimage.uniform_filter1d(raw**2, self.samples_ciclo, axis=0)
                ))
            except Exception:
                sinal['rms'] = np.zeros_like(raw)
            if nome.startswith('I_'):
                try:
                    sinal['seq'] = self._calcular_seq(raw)
                except Exception:
                    sinal['seq'] = None
    
    def _calcular_seq(self, abc):
        a = np.exp(1j * 2*np.pi/3)
        rot = np.exp(-1j * 2*np.pi*60*self.t)
        fasores = np.zeros((len(self.t), 3), dtype=complex)
        for i in range(3):
            fasores[:, i] = scipy.ndimage.uniform_filter1d(
                abc[:, i] * rot, self.samples_ciclo
            ) * np.sqrt(2)
        I0 = (fasores[:,0] + fasores[:,1] + fasores[:,2])/3
        I1 = (fasores[:,0] + a*fasores[:,1] + a**2*fasores[:,2])/3
        I2 = (fasores[:,0] + a**2*fasores[:,1] + a*fasores[:,2])/3
        return {'I0': np.abs(I0), 'I1': np.abs(I1), 'I2': np.abs(I2)}
    
    def detectar_falta(self, sinal_preferencial='I_822'):
        if sinal_preferencial not in self.dados or self.dados[sinal_preferencial]['rms'] is None:
            for nome, info in self.dados.items():
                if nome.startswith('I_') and info['rms'] is not None:
                    sinal_preferencial = nome
                    break
            else:
                return len(self.t)//2
        rms = self.dados[sinal_preferencial]['rms']
        i_max = np.max(rms, axis=1)
        baseline = np.median(i_max[:len(i_max)//5])
        idx = np.where(i_max > baseline*2.0)[0]
        return idx[0] if len(idx) > 0 else len(self.t)//2

In [ ]:
# ======================================================================
# ANALISADOR PRINCIPAL – METADADOS E EXTRAÇÃO DE MÉTRICAS
# ======================================================================

class AnalisadorCapitulo4:
    def __init__(self, pasta_entrada, pasta_saida='Resultados_Cap4_SWER_T2F'):
        self.pasta = Path(pasta_entrada)
        self.pasta_saida = Path(pasta_saida)
        self.arquivos = sorted(self.pasta.glob('*_py.mat'))
        self.resultados = []
        self.df = None
        
        for p in [
            self.pasta_saida / 'CSVs',
            self.pasta_saida / 'Graficos' / 'MRT_COM_AT',
            self.pasta_saida / 'Graficos' / 'MRT_SEM_AT',
            self.pasta_saida / 'Graficos' / 'T2F_COM_AT',
            self.pasta_saida / 'Graficos' / 'T2F_SEM_AT',
        ]:
            p.mkdir(parents=True, exist_ok=True)
    
    def processar_todos(self):
        for arq in self.arquivos:
            try:
                proc = ProcessadorSinais(str(arq))
                self.resultados.append(self._extrair_metricas(proc, arq.stem))
            except Exception as e:
                print("ERRO:", e)
        self.df = pd.DataFrame(self.resultados) if self.resultados else pd.DataFrame()
        return self.df
    
    def _extrair_metricas(self, proc, nome_arq):
        partes = nome_arq.split('__')
        simulacao = partes[0]
        caso = partes[1] if len(partes) > 1 else ''
        
        if simulacao.startswith('MRT'):
            tipo_sistema = 'MRT'
        else:
            tipo_sistema = 'T2F'
        
        sem_aterramento = ('sem_terra' in simulacao) or ('no_Ground' in simulacao)
        
        if ('Sem_Falta' in caso) or ('Normal' in caso):
            tipo_falta = 'Normal'
        elif 'Falta_ABC' in caso:
            tipo_falta = 'ABC'
        elif 'Falta_AB' in caso:
            tipo_falta = 'AB'
        elif 'Falta_AC' in caso:
            tipo_falta = 'AC'
        elif 'Falta_BC' in caso:
            tipo_falta = 'BC'
        elif 'Falta_A' in caso:
            tipo_falta = 'A-G'
        else:
            tipo_falta = 'Desconhecido'
        
        local = 'Indefinido'
        for loc in ['822_Meio', '820_Meio', '818_2_Meio',
                    '822', '820', '818_2', '818_1', '816']:
            if loc in caso:
                local = f'Barra {loc.replace("_", ".")}'
                break
        
        idx_falta = proc.detectar_falta('I_822')
        
        resultado = {
            'arquivo': nome_arq,
            'tipo_sistema': tipo_sistema,
            'sem_aterramento': sem_aterramento,
            'condicao_aterramento': 'SEM Aterramento' if sem_aterramento else 'COM Aterramento',
            'tipo_falta': tipo_falta,
            'local_falta': local,
            'm1': float(proc.m1),
            't_falta': float(proc.t[idx_falta]),
            'processador': proc
        }
        
        for ponto in ['800', '816', '818', '820', '822']:
            nome_i = f'I_{ponto}'
            if nome_i in proc.dados and proc.dados[nome_i]['rms'] is not None:
                rms = proc.dados[nome_i]['rms']
                resultado[f'{ponto}_I_pico_A'] = float(np.max(rms[:, 0]))
                resultado[f'{ponto}_I_pico_B'] = float(np.max(rms[:, 1]))
                resultado[f'{ponto}_I_pico_C'] = float(np.max(rms[:, 2]))
                resultado[f'{ponto}_I_pico_max'] = float(np.max(rms))
                if proc.dados[nome_i]['seq']:
                    seq = proc.dados[nome_i]['seq']
                    resultado[f'{ponto}_I0'] = float(seq['I0'][idx_falta])
                    resultado[f'{ponto}_I1'] = float(seq['I1'][idx_falta])
                    resultado[f'{ponto}_I2'] = float(seq['I2'][idx_falta])
        return resultado


In [ ]:
# ======================================================================
# MÉTODOS DO ANALISADOR – TABELAS (CSV + LaTeX)
# ======================================================================

def gerar_tabelas_comparativas(self):
    if self.df is None or len(self.df) == 0:
        return
    
    print(f"\n{'GERANDO TABELAS COMPARATIVAS':^100}")
    print(f"{'-'*100}\n")
    
    # Tabela 0 – Casos de simulação
    cols_meta = ['arquivo', 'tipo_sistema', 'condicao_aterramento',
                 'tipo_falta', 'local_falta', 'm1']
    df_casos = self.df[cols_meta].sort_values(
        ['tipo_sistema', 'condicao_aterramento', 'tipo_falta', 'local_falta', 'm1']
    )
    df_casos.to_csv(self.pasta_saida / 'CSVs' / 'Tabela_0_Casos_Simulacao.csv',
                    index=False, encoding='utf-8-sig')
    
    # Tabela 1 – Resumo geral barra 822
    tabela1_data = []
    for sistema in ['MRT', 'T2F']:
        df_sistema = self.df[self.df['tipo_sistema'] == sistema]
        for tipo_falta in df_sistema['tipo_falta'].unique():
            if tipo_falta in ['Desconhecido', 'Normal']:
                continue
            df_tipo = df_sistema[df_sistema['tipo_falta'] == tipo_falta]
            if len(df_tipo) > 0 and '822_I_pico_max' in df_tipo.columns:
                tabela1_data.append({
                    'Sistema': sistema,
                    'Tipo de Falta': tipo_falta,
                    'N° Casos': len(df_tipo),
                    'I_pico_min (A)': df_tipo['822_I_pico_max'].min(),
                    'I_pico_max (A)': df_tipo['822_I_pico_max'].max(),
                    'I_pico_média (A)': df_tipo['822_I_pico_max'].mean(),
                    'I_pico_std (A)': df_tipo['822_I_pico_max'].std(),
                    'I0_média (A)': df_tipo['822_I0'].mean() if '822_I0' in df_tipo.columns else 0,
                    'I1_média (A)': df_tipo['822_I1'].mean() if '822_I1' in df_tipo.columns else 0,
                    'I2_média (A)': df_tipo['822_I2'].mean() if '822_I2' in df_tipo.columns else 0
                })
    df_tab1 = pd.DataFrame(tabela1_data)
    df_tab1.to_csv(self.pasta_saida / 'CSVs' / 'Tabela_1_Resumo_Geral.csv',
                   index=False, encoding='utf-8-sig', float_format='%.2f')
    with open(self.pasta_saida / 'Tabelas_LaTeX' / 'Tabela_1_Resumo_Geral.tex',
              'w', encoding='utf-8') as f:
        f.write("\\begin{table}[htbp]\n")
        f.write("\\centering\n")
        f.write("\\caption{Resumo estatístico das correntes de curto-circuito na barra 822}\n")
        f.write("\\label{tab:resumo_geral_cap4}\n")
        f.write(df_tab1.to_latex(index=False, float_format='%.2f'))
        f.write("\\end{table}\n")
    
    # Tabela 2 – Comparação MRT A-G vs T2F (ABC, AB, AC, BC)
    tabela2_data = []
    for com_terra in [True, False]:
        terra_label = 'COM Aterramento' if com_terra else 'SEM Aterramento'
        df_mrt = self.df[
            (self.df['tipo_sistema'] == 'MRT') &
            (self.df['tipo_falta'] == 'A-G') &
            (self.df['sem_aterramento'] == (not com_terra)) &
            (np.abs(self.df['m1'] - 0.5) < 0.05)
        ]
        for tipo_t2f in ['ABC', 'AB', 'AC', 'BC']:
            df_t2f = self.df[
                (self.df['tipo_sistema'] == 'T2F') &
                (self.df['tipo_falta'] == tipo_t2f) &
                (self.df['sem_aterramento'] == (not com_terra)) &
                (np.abs(self.df['m1'] - 0.5) < 0.05)
            ]
            if len(df_mrt) > 0 and len(df_t2f) > 0:
                for col, label in [('822_I_pico_max', 'I_pico'),
                                   ('822_I0', 'I0'),
                                   ('822_I1', 'I1'),
                                   ('822_I2', 'I2')]:
                    if col in df_mrt.columns and col in df_t2f.columns:
                        val_mrt = df_mrt[col].mean()
                        val_t2f = df_t2f[col].mean()
                        delta_abs = val_t2f - val_mrt
                        delta_pct = (delta_abs / val_mrt * 100) if val_mrt != 0 else 0
                        tabela2_data.append({
                            'Condição': terra_label,
                            'Falta_MRT': 'A-G',
                            'Falta_T2F': tipo_t2f,
                            'Métrica': label,
                            'MRT (A)': val_mrt,
                            'T2F (A)': val_t2f,
                            'Δ Absoluto (A)': delta_abs,
                            'Δ Relativo (%)': delta_pct
                        })
    df_tab2 = pd.DataFrame(tabela2_data)
    df_tab2.to_csv(self.pasta_saida / 'CSVs' / 'Tabela_2_Comparacao_MRT_vs_T2F.csv',
                   index=False, encoding='utf-8-sig', float_format='%.2f')
    with open(self.pasta_saida / 'Tabelas_LaTeX' / 'Tabela_2_Comparacao_MRT_vs_T2F.tex',
              'w', encoding='utf-8') as f:
        f.write("\\begin{table}[htbp]\n")
        f.write("\\centering\n")
        f.write("\\caption{Comparação quantitativa entre MRT (A-G) e T2F na barra 822, m1 \\approx 0{,}5}\n")
        f.write("\\label{tab:comparacao_t2f_mrt_cap4}\n")
        f.write(df_tab2.to_latex(index=False, float_format='%.2f'))
        f.write("\\end{table}\n")

    print("✓ Tabelas geradas.")

# anexa o método à classe
AnalisadorCapitulo4.gerar_tabelas_comparativas = gerar_tabelas_comparativas


In [ ]:
# ======================================================================
# MÉTODOS DO ANALISADOR – GRÁFICOS E PDF
# ======================================================================

def gerar_graficos_profissionais(self):
    if self.df is None or len(self.df) == 0:
        return
    print(f"\n{'GERANDO GRÁFICOS PROFISSIONAIS':^100}")
    print(f"{'-'*100}\n")
    
    # MRT (SWER) A-G
    print("  Gerando gráficos MRT (SWER)...")
    df_mrt = self.df[(self.df['tipo_sistema'] == 'MRT') &
                     (self.df['tipo_falta'] == 'A-G')]
    for com_terra in [True, False]:
        df_sub = df_mrt[df_mrt['sem_aterramento'] == (not com_terra)]
        if len(df_sub) < 2:
            continue
        terra_label = 'COM_Aterramento' if com_terra else 'SEM_Aterramento'
        pasta_fig = self.pasta_saida / 'Graficos' / ('MRT_COM_AT' if com_terra else 'MRT_SEM_AT')
        for local in sorted(df_sub['local_falta'].unique()):
            df_loc = df_sub[df_sub['local_falta'] == local].sort_values('m1')
            if len(df_loc) < 2:
                continue
            fig = plt.figure(figsize=(18, 12))
            gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.25)
            ax1 = fig.add_subplot(gs[0, :])
            ax1.plot(df_loc['m1']*100, df_loc['822_I_pico_A'], 'o-', color=CORES['Fase_A'],
                     label='Fase A (faltosa)', linewidth=3, markersize=10,
                     markeredgecolor='white', markeredgewidth=1.5)
            ax1.plot(df_loc['m1']*100, df_loc['822_I_pico_B'], 's--', color=CORES['Fase_B'],
                     label='Fase B (sã)', linewidth=2, markersize=8, alpha=0.7)
            ax1.plot(df_loc['m1']*100, df_loc['822_I_pico_C'], '^--', color=CORES['Fase_C'],
                     label='Fase C (sã)', linewidth=2, markersize=8, alpha=0.7)
            ax1.set_xlabel('Posição da falta (%)')
            ax1.set_ylabel('I_pico RMS (A)')
            ax1.set_title(f'MRT (SWER) - Falta A-G ({terra_label}) - {local}')
            ax1.legend(); ax1.grid(True, alpha=0.3, linestyle='--')
            
            ax2 = fig.add_subplot(gs[1, :])
            if '822_I0' in df_loc.columns:
                ax2.plot(df_loc['m1']*100, df_loc['822_I0'], 'o-', color=CORES['I0'], label='I₀')
                ax2.plot(df_loc['m1']*100, df_loc['822_I1'], 's-', color=CORES['I1'], label='I₁')
                ax2.plot(df_loc['m1']*100, df_loc['822_I2'], '^-', color=CORES['I2'], label='I₂')
            ax2.set_xlabel('Posição da falta (%)')
            ax2.set_ylabel('I seq (A)')
            ax2.set_title('Componentes simétricas'); ax2.legend(); ax2.grid(True, alpha=0.3, linestyle='--')
            
            ax3 = fig.add_subplot(gs[2, 0])
            if '822_V_min' in df_loc.columns:
                ax3.plot(df_loc['m1']*100, df_loc['822_V_min'], 'o-', color='#E74C3C', label='V_min')
                ax3.plot(df_loc['m1']*100, df_loc['822_V_max'], 's-', color='#27AE60', label='V_max')
            ax3.set_xlabel('Posição da falta (%)')
            ax3.set_ylabel('V (V)')
            ax3.set_title('Afundamento de tensão'); ax3.legend(); ax3.grid(True, alpha=0.3, linestyle='--')
            
            ax4 = fig.add_subplot(gs[2, 1])
            if '822_I0' in df_loc.columns and '822_I1' in df_loc.columns:
                razao = df_loc['822_I0'] / (df_loc['822_I1'] + 1e-6)
                ax4.plot(df_loc['m1']*100, razao, 'o-', color='#9B59B6')
                ax4.axhline(1.0, color='red', linestyle='--', alpha=0.7)
            ax4.set_xlabel('Posição da falta (%)')
            ax4.set_ylabel('I₀/I₁')
            ax4.set_title('Caracterização A-G'); ax4.grid(True, alpha=0.3, linestyle='--')
            
            plt.suptitle(f'Sistema MRT (SWER) - {terra_label} - {local}', fontsize=16)
            nome_fig = f'MRT_AG_{terra_label}_{local.replace(" ", "_")}.png'
            plt.savefig(pasta_fig / nome_fig, dpi=300, bbox_inches='tight', facecolor='white')
            plt.close()
            print("    ✓", nome_fig)
    
    # T2F
    print("\n  Gerando gráficos T2F...")
    df_t2f = self.df[self.df['tipo_sistema'] == 'T2F']
    for tipo_falta in ['ABC', 'AB', 'AC', 'BC']:
        df_falta = df_t2f[df_t2f['tipo_falta'] == tipo_falta]
        if len(df_falta) == 0:
            continue
        for com_terra in [True, False]:
            df_sub = df_falta[df_falta['sem_aterramento'] == (not com_terra)]
            if len(df_sub) < 2:
                continue
            terra_label = 'COM_Aterramento' if com_terra else 'SEM_Aterramento'
            pasta_fig = self.pasta_saida / 'Graficos' / ('T2F_COM_AT' if com_terra else 'T2F_SEM_AT')
            for local in sorted(df_sub['local_falta'].unique()):
                df_loc = df_sub[df_sub['local_falta'] == local].sort_values('m1')
                if len(df_loc) < 2:
                    continue
                fig = plt.figure(figsize=(18, 12))
                gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.25)
                ax1 = fig.add_subplot(gs[0, :])
                ax1.plot(df_loc['m1']*100, df_loc['822_I_pico_A'], 'o-', color=CORES['Fase_A'], label='A')
                ax1.plot(df_loc['m1']*100, df_loc['822_I_pico_B'], 's-', color=CORES['Fase_B'], label='B')
                ax1.plot(df_loc['m1']*100, df_loc['822_I_pico_C'], '^-', color=CORES['Fase_C'], label='C')
                ax1.set_xlabel('Posição da falta (%)')
                ax1.set_ylabel('I_pico RMS (A)')
                ax1.set_title(f'T2F - Falta {tipo_falta} ({terra_label}) - {local}')
                ax1.legend(); ax1.grid(True, alpha=0.3, linestyle='--')
                
                ax2 = fig.add_subplot(gs[1, :])
                if '822_I0' in df_loc.columns:
                    ax2.plot(df_loc['m1']*100, df_loc['822_I0'], 'o-', color=CORES['I0'], label='I₀')
                    ax2.plot(df_loc['m1']*100, df_loc['822_I1'], 's-', color=CORES['I1'], label='I₁')
                    ax2.plot(df_loc['m1']*100, df_loc['822_I2'], '^-', color=CORES['I2'], label='I₂')
                ax2.set_xlabel('Posição da falta (%)')
                ax2.set_ylabel('I seq (A)')
                ax2.set_title('Componentes simétricas'); ax2.legend(); ax2.grid(True, alpha=0.3, linestyle='--')
                
                ax3 = fig.add_subplot(gs[2, 0])
                if '822_I_pico_A' in df_loc.columns:
                    i_avg = (df_loc['822_I_pico_A'] +
                             df_loc['822_I_pico_B'] +
                             df_loc['822_I_pico_C']) / 3
                    deseq = np.abs(df_loc['822_I_pico_max'] - i_avg) / (i_avg + 1e-6) * 100
                    ax3.plot(df_loc['m1']*100, deseq, 'o-', color='#E67E22')
                ax3.set_xlabel('Posição da falta (%)')
                ax3.set_ylabel('Desequilíbrio (%)')
                ax3.set_title('Desequilíbrio entre fases'); ax3.grid(True, alpha=0.3, linestyle='--')
                
                ax4 = fig.add_subplot(gs[2, 1])
                ax4.plot(df_loc['m1']*100, df_loc['822_I_pico_max'], 'o-', color=CORES['T2F'])
                ax4.set_xlabel('Posição da falta (%)')
                ax4.set_ylabel('I_pico_max (A)')
                ax4.set_title('Corrente máxima de falta'); ax4.grid(True, alpha=0.3, linestyle='--')
                
                plt.suptitle(f'Sistema T2F - Falta {tipo_falta} - {terra_label} - {local}', fontsize=16)
                nome_fig = f'T2F_{tipo_falta}_{terra_label}_{local.replace(" ", "_")}.png'
                plt.savefig(pasta_fig / nome_fig, dpi=300, bbox_inches='tight', facecolor='white')
                plt.close()
                print("    ✓", nome_fig)

AnalisadorCapitulo4.gerar_graficos_profissionais = gerar_graficos_profissionais


In [ ]:
pasta_entrada = Path("C:/Users/Leonardo Felipe/OneDrive/Coisas_Leonardo/gits/CurtosT2F/T2F_MATLAB/NovoArtigoPowerDelivery34bus/NovoModeloQualificacao/Teste_Novo_Sem_Terra_10/Processados_HDF5")

In [ ]:
pasta_saida = "Resultados_Cap4_SWER_T2F"

In [ ]:
analisador = AnalisadorCapitulo4(pasta_entrada, pasta_saida)
df = analisador.processar_todos()
analisador.gerar_tabelas_comparativas()
analisador.gerar_graficos_profissionais()
